# အခန်း ၀၁ - AI အေးဂျင့်များအတွက် နိဒါန်း

**AI အေးဂျင့်များ အစပျိုးသူများအတွက်** သင်တန်း၏ ပထမဆုံးအခန်းကို ကြိုဆိုပါတယ်!

**AI အေးဂျင့်** ဆိုတာသည် အကြီးစား ဘာသာစကားမော်ဒယ် (LLM) ကို ၎င်း၏ အတွေးအခေါ်စနစ်အဖြစ် အသုံးပြု၍၊ အသုံးပြုသူအတွက် ရည်မှန်းချက်တစ်ခုကို ပြည့်မီစေရန် ကမ္ဘာပေါ်တွင် *ဆောင်ရွက်မှုများ* — API များကို ခေါ်ယူခြင်း၊ ဒေတာဘေ့စ်များကို စုံစမ်းမေးမြန်းခြင်း သို့မဟုတ် ကုဒ်များ chạy လုပ်ခြင်းတို့ ပြုလုပ်နိုင်သော ပရိုဂရမ်တစ်ခုဖြစ်သည်။

ဤမှတ်စုစာအုပ်တွင် သင့်ပထမဆုံးအေးဂျင့်ဖြစ်သည့် **ခရီးသွားအေးဂျင့်** ကို တည်ဆောက်မည်ဖြစ်ပြီး၊ ၎င်းသည် ခရီးစရာနေရာများကို အကြံပြုရာ ပါဝင်သည်။ လမ်းကြောင်းအတိုင်း သင်သည် အောက်ပါအရာများကို သင်ယူမည်ဖြစ်သည်။

၁။ **Microsoft Agent Framework** ကို အသုံးပြု၍ Microsoft Foundry Agent Service နှင့် ချိတ်ဆက်ခြင်း။
၂။ အေးဂျင့်ကို **ကိရိယာ** တစ်ခုပေးခြင်း — ၎င်းက ခေါ်နိုင်သော ရိုးရှင်းသော Python function တစ်ခုဖြစ်သည်။
၃။ အေးဂျင့်ကို လည်ပတ်စေပြီး ၎င်း၏တုံ့ပြန်မှုကို စစ်ဆေးခြင်း။
၄။ အေးဂျင့်၏ တုံ့ပြန်မှုကို token-တစ်ခုစီအလိုက် စီးရီးလိုက် streams စနစ်ဖြင့် ဖော်ပြခြင်း။


## စတင်ခြင်း

ဒီ notebook ကို လည်ပတ်ရန်မတိုင်မီ သေချာရန်ရှိသည်မှာ-

၁။ **Microsoft Foundry project တစ်ခု** ထည့်သွင်းထားပြီး chat model တစ်ခု (ဥပမာ `gpt-4.1-mini`) ကို deploy ပြုလုပ်ထားရမည်။
၂။ **Azure CLI နှင့် login ပြုလုပ်ထားပြီး** — terminal တွင် `az login` အမိန့်ကို run ပြုလုပ်ပါ။
၃။ **လိုအပ်သည့် ပတ်ဝန်းကျင် အပြောင်းအလဲများကို သတ်မှတ်ထားရမည် -**
   - `AZURE_AI_PROJECT_ENDPOINT` — သင့် Microsoft Foundry project endpoint။
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — deploy ပြုလုပ်ထားသော model ၏ အမည်။

အောက်ပါ cell တွင် သင်လိုအပ်သော Python packages များကို install ပြုလုပ်ပါမည်။


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## သင့်ရဲ့ ပထမဆုံး Agent ကို တည်ဆောက်ခြင်း

Agent တစ်ယောက်အတွက် လိုအပ်တာ နှစ်ခု ရှိပါတယ်။

- **ညွှန်ကြားချက်များ** သည် အဲဒီ agent ရဲ့ *သူက ဘယ့်သူလဲ* နှင့် *ဘယ်လို အပြုအမူ ပြုလုပ်သင့်သည်* ဆိုတာကို ပြောပြတဲ့ (system prompt) ပါ။
- **ကိရိယာများ** — Python function များဖြစ်ပြီး `@tool` နဲ့ အလှူခံထားသော၊ agent အနေဖြင့် အချက်အလက် ရယူရန် သို့မဟုတ် လုပ်ဆောင်ချက်များ အတက် လှူနိုင်သော function များ ဖြစ်သည်။

အောက်ပါတွင် ကျွန်ုပ်တို့သည် လူကြိုက်များသော ခရီးသွားရာ အနေအထားများစာရင်းကို ပြန်ပေးသည့် ရိုးရှင်းသော ကိရိယာတစ်ခု ကို သတ်မှတ်ထားသည်။ အသုံးပြုသူက ခရီးသို့ သွားရန် အကြံပြုချက် တောင်းဆိုသည့်အခါ agent သည် ကိရိယာကို အသုံးပြုပါမည်။


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## တိုက်ရိုက်တုံ့ပြန်ချက်များ

ပိုမို ပေါ့ပါး သော အတွေ့အကြုံအတွက်၊ သင့်အေးဂျင့်၏ တုံ့ပြန်ချက်ကို **တိုက်ရိုက်ထုတ်လွှင့်** သွားနိုင်သည်။ အပြည့်အစုံ တုံ့ပြန်ချက်ကို စောင့်ဆိုင်းမှုမဟုတ်ပဲ၊ အေးဂျင့်သည် စာသားပိုင်းများကို ထုတ်လွှင့်ပေးသည်။ ၎င်းသည် စကားပြော အင်တာဖေ့စ်များတွင် အထူးသဖြင့် အသုံးဝင်ပြီး အချိန်နှင့်တပြေးညီ ထွက်ရှိသော အချက်အလက်များကို ပြသလိုသော အခါတွင် အသုံးဝင်သည်။


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## အကျဉ်းချုပ်

ဤသင်ခန်းစာတွင် သင် သင်ယူခဲ့သည်မှာ -

- **Microsoft Foundry Agent Service နှင့် ချိတ်ဆက်ရန် `FoundryChatClient` ဖြင့် provider တစ်ခု ဖန်တီးပါ**။
- **အေးဂျင့်သည် သင့် Python function များကို ခေါ်နိုင်ရန်အတွက် `@tool` decorator ဖြင့် tool တစ်ခု သတ်မှတ်ပါ**။
- **အသုံးပြုသူအတွက် message ဖြင့် အေးဂျင့်ကို လည်ပတ်ပြီး ၎င်း၏ တုံ့ပြန်မှုကို ပရင့်ထုတ်ပါ**။
- **တုံ့ပြန်မှုများကို တိုက်ရိုက်ထွက်ရှိရန် stream လုပ်ပါ**။

နောက်ဆုံးသင်ခန်းစာတွင် agentic frameworks များကို ပိုမိုနက်ရှိုင်းစေပြီး အေးဂျင့်များအား ပိုမို အားကောင်းသည့် tools များနှင့် multi-step reasoning စွမ်းရည်များ ပေးပို့ပုံကို သင်ယူမည်ဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
